In [ ]:
# --- Act 0: Setup (hidden) ---
import subprocess
import sys
import warnings

warnings.filterwarnings('ignore')

# The user IPython profile stubs sys.modules['kaleido']=None on ARM/Tegra as an
# old kaleido-0.x SIGABRT guard.  kaleido 1.x renders static PNGs via
# Chrome/choreographer and does not import tensorflow/jax, so un-stub it here so
# the plotly charts export under nbconvert regardless of the active profile.
for _stubbed in ('kaleido', 'kaleido.scopes', 'kaleido.scopes.plotly'):
    if sys.modules.get(_stubbed) is None:
        del sys.modules[_stubbed]
try:  # reset plotly's cached availability probe if it ran before the un-stub
    import plotly.io._kaleido as _pk
    _pk._KALEIDO_AVAILABLE = None
    _pk._KALEIDO_MAJOR = None
except Exception:
    pass

import html
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
from textwrap import shorten

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from scipy import stats

# --- CWD-robust repository-root resolution ---
# The preprint build executes a COPY of this notebook with CWD =
# notebooks/preprint/, so REPO_ROOT must not depend on the CWD depth.  Walk up
# from the current directory to the repo marker (pyproject.toml + engine/); fall
# back to `git rev-parse --show-toplevel`.  Never use a fixed relative depth.
def _find_repo_root() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / 'pyproject.toml').exists() and (base / 'engine').is_dir():
            return base
    try:
        top = subprocess.check_output(
            ['git', 'rev-parse', '--show-toplevel'], text=True
        ).strip()
        if top:
            return Path(top)
    except Exception:
        pass
    raise FileNotFoundError(
        "Cannot locate repository root (no pyproject.toml with engine/ walking "
        "up from CWD, and `git rev-parse --show-toplevel` failed)."
    )

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026'
RARR_CYCLE = REPO_ROOT / 'projects' / 'owasp-llm' / 'cycles' / '2026-rarr'
BASELINES = REPO_ROOT / 'projects' / 'owasp-llm' / 'baselines' / '2026'

# --- Preprint figure output dir (300-dpi PNGs saved by the chart library) ---
PREPRINT_FIG = REPO_ROOT / 'notebooks' / 'preprint' / 'figures'
PREPRINT_FIG.mkdir(parents=True, exist_ok=True)

# --- Tested data + chart library (remediation #1: one tested chart library) ---
from engine.report.narrative_data import load_narrative_data
from engine.report.narrative_charts import (
    render_stratum_bar,
    render_tier_donut,
    render_confusion_heatmap,
    render_precision_bars,
    render_precision_posteriors,
    render_ridge_plot,
    render_dumbbell_chart,
    render_plotly_rankings,
    render_bump_chart,
    render_ci_overlap,
    render_paired_dots,
    render_theme_bars,
    render_oos_treemap,
    render_sankey_confusion,
    render_confusion_matrix_3x3,
    render_rank_change_2025_2026,
    render_entry_expansion_map,
    render_rarr_robustness,
)
from engine.report.blend_2025_2026 import (
    blended_ranking,
    load_entries,
    rank_moves,
)

# --- Seaborn / matplotlib theme ---
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
})

FRAME_BLIND = {'LLM04', 'LLM08', 'LLM10'}

# --- Deep-dive sidebar helpers (used by the narrative markdown/HTML cells) ---
def sidebar(title, content):
    """Render a collapsible deep-dive sidebar."""
    return HTML(
        f'<details style="margin: 1em 0; padding: 0.5em; '
        f'border-left: 3px solid #4a86c8; background: #f8f9fa;">'
        f'<summary style="cursor: pointer; font-weight: bold; '
        f'color: #2c5282;">{title}</summary>'
        f'<div style="margin-top: 0.8em; line-height: 1.6;">{content}</div>'
        f'</details>'
    )

def callout_box(text, border_color='#e53e3e', bg_color='#fff5f5'):
    """Render an always-visible boxed callout (not collapsed)."""
    return HTML(
        f'<div style="margin: 1em 0; padding: 1em; border-left: 4px solid {border_color}; '
        f'background: {bg_color}; line-height: 1.6;">{text}</div>'
    )

# --- Load all pre-computed data via the tested loader ---
DATA = load_narrative_data(CYCLE)
ENTRY_NAMES = DATA['entry_names']
INFER_ENTRY_ORDER = DATA['entry_ids']

# Validate shapes
assert DATA['lambda_samples'].shape == (16000, 20), (
    f"Expected lambda_samples shape (16000, 20), got {DATA['lambda_samples'].shape}"
)
assert len(INFER_ENTRY_ORDER) == 20, (
    f"Expected 20 entry_ids in inference_summary, got {len(INFER_ENTRY_ORDER)}"
)

print(f"Loaded data via load_narrative_data. {len(DATA['incidents'])} incidents, "
      f"{len(DATA['prelabels'])} prelabels, {len(DATA['goldset'])} adjudications, "
      f"{DATA['lambda_samples'].shape[0]} posterior draws. "
      f"Figures -> {PREPRINT_FIG}")


# Part I

<!-- PLACEHOLDER — prose author fills this section; no narrative in the code pass. -->


# What the Data Says About the 2026 Top 10

## Act 1: The Question

The OWASP Top 10 for LLMs ranks AI security vulnerabilities. The 2025 list was built
from expert surveys — hundreds of security professionals voting on what matters most.
That process produced a consensus: Prompt Injection at #1, Sensitive Information
Disclosure at #2, and so on down to #10.

Expert opinion is one signal. We wanted to check it against a second signal:
the pattern of real-world incidents. We built a corpus of ~6,600 AI security incidents
from public databases, classified each one against the 20-entry taxonomy, and asked:
does the incident data agree with the experts?

This notebook walks through that analysis step by step. Along the way, you will see
how the classification worked, how we measured its accuracy, and what a Bayesian model
does with noisy measurements. Every chart and table below is computed live from the
data — you can re-run any cell to verify.

Here are the 20 taxonomy entries we are working with. The "Incident Rank" column is
blank for now. We will fill it in Act 6, after walking through the methodology.

In [ ]:
# Build the entry table with a placeholder rank column
entry_table = pd.DataFrame([
    {'#': i + 1, 'Entry ID': e['entry_id'], 'Name': e['canonical_name'], 'Incident Rank': '—'}
    for i, e in enumerate(DATA['rubric']['entries'])
])
entry_table = entry_table.set_index('#')
display(entry_table.style.set_caption(
    "20 taxonomy entries. Incident Rank will be filled in Act 6."
).set_properties(**{'text-align': 'left'}))

## Act 2: The Corpus

The corpus contains 6,639 incidents from public databases: CVE, GHSA, and OSV
(security advisories), plus AIAAIC (a database of AI-related harms and controversies).
Each record has a text description of what happened.

The corpus splits into two strata. The **security** stratum (CVE/GHSA/OSV) contains
things like prompt injection exploits, data leakage through APIs, and supply chain
compromises in ML packages. The **ai-harm** stratum (AIAAIC) contains things like
algorithmic discrimination, deepfake misuse, and surveillance overreach.

This split matters. The classifier performs differently on each stratum — security
incidents have more structured descriptions (CVE format), while ai-harm incidents
are written as news summaries with varying detail.

In [ ]:
# Act 2: incidents-by-stratum bar chart (saves stratum_bar.png @ 300 dpi)
render_stratum_bar(DATA, PREPRINT_FIG)


In [ ]:
# Show 2 real incident examples — one from each stratum
# Build stratum lookup from classified incidents (prelabels has no stratum field)
_stratum_lookup = {inc['incident_id']: inc['stratum'] for inc in DATA['incidents']}

prelabels_df = DATA['prelabels']
security_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                    and p['consensus'] not in ('out-of-scope', None)
                    and _stratum_lookup.get(p['incident_id']) == 'security'), None)
harm_ex = next((p for p in prelabels_df if p['triage_tier'] == 'agree'
                and _stratum_lookup.get(p['incident_id']) == 'ai-harm'
                and len(p['text']) > 100), None)
if security_ex is None or harm_ex is None:
    display(HTML('<p style="color: red;">Could not find suitable example incidents.</p>'))

def incident_card(record, label):
    """Format an incident as an HTML card."""
    text = html.escape(shorten(record['text'], width=250, placeholder='...'))
    consensus = html.escape(str(record['consensus']))
    tier = html.escape(str(record['triage_tier']))
    inc_id = html.escape(str(record['incident_id']))
    return HTML(
        f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
        f'margin: 0.5em 0; background: #fafafa;">'
        f'<strong>{html.escape(label)}</strong><br>'
        f'<code>{inc_id}</code> · consensus: '
        f'<strong>{consensus}</strong> · tier: {tier}<br>'
        f'<p style="margin-top: 0.5em; color: #333;">{text}</p>'
        f'</div>'
    )

display(HTML('<h4>Example: Security stratum (CVE/GHSA/OSV)</h4>'))
display(incident_card(security_ex, 'Security incident'))

display(HTML('<h4>Example: AI-harm stratum (AIAAIC)</h4>'))
display(incident_card(harm_ex, 'AI-harm incident'))

In [ ]:
# F-frame sidebar
display(sidebar(
    'Deep dive: What the corpus cannot see (F-frame)',
    '<p>The corpus is built from a keyword crawl of public databases — CVE, GHSA, OSV, '
    'and AIAAIC. Incidents that never became CVEs or harm-database entries are invisible '
    'to us. A prompt injection attack against an internal enterprise tool that was caught '
    'and patched quietly will never appear in this data.</p>'
    '<p>This creates structural bias toward vulnerability types that get reported in '
    'public channels. Well-resourced organizations that fix issues internally are '
    'underrepresented. Novel attack types that have not been assigned a CVE category '
    'are invisible.</p>'
))

# F-circ callout — always visible, not collapsed
display(callout_box(
    '<strong>Structural limitation: taxonomy-frame circularity (F-circ)</strong><br><br>'
    'We classified these incidents using the same taxonomy we are trying to validate. '
    'If the classifier systematically favors certain entries, the incident counts will '
    'appear to confirm the expert rankings even if the true pattern is different. '
    'This is taxonomy-frame circularity. It means the concordance we measure later '
    'is an upper bound on true agreement, not a precise estimate of it.',
    border_color='#d69e2e', bg_color='#fefcbf'
))

## Act 3: Classification — How We Labeled 6,600 Incidents

Each incident was classified by three different large language models: Qwen 235B,
Llama 405B, and DeepSeek V3. Each model independently read the incident text and
assigned it to one of the 20 taxonomy entries — or marked it "out of scope" if none
fit.

When all three models agreed on the same entry, we call it **agree tier**. When two
agreed and one differed, **split tier**. When all three picked different entries,
**disagree tier**.

The tier tells us how confident we can be in the classification. Agree-tier incidents
have strong consensus. Disagree-tier incidents sit in ambiguous territory where even
three independent classifiers could not converge.

In [ ]:
# Find a good agree-tier example with three matching votes
agree_ex = next((p for p in DATA['prelabels']
                 if p['triage_tier'] == 'agree'
                 and p['consensus'] not in ('out-of-scope', None)
                 and len(p['model_votes']) == 3
                 and len(p['text']) > 120), None)
assert agree_ex is not None, "No suitable agree-tier example found in prelabels"

display(HTML('<h4>Worked example: three-model classification</h4>'))
display(HTML(
    f'<div style="border: 1px solid #ccc; border-radius: 8px; padding: 1em; '
    f'margin: 0.5em 0; background: #f7fafc;">'
    f'<p style="color: #555;"><strong>Incident text</strong> (truncated):</p>'
    f'<p style="font-style: italic;">{html.escape(shorten(agree_ex["text"], 250, placeholder="..."))}</p>'
    f'<hr style="border: 0; border-top: 1px solid #e2e8f0;">'
    f'<table style="width: 100%; border-collapse: collapse;">'
    f'<tr style="background: #edf2f7;"><th>Model</th><th>Entry</th><th>Confidence</th></tr>'
    + ''.join(
        f'<tr><td>{html.escape(v["model_id"].split("/")[-1])}</td>'
        f'<td><strong>{html.escape(v["entry_id"])}</strong></td>'
        f'<td>{v["confidence"]:.0%}</td></tr>'
        for v in agree_ex['model_votes']
    )
    + f'</table>'
    f'<p style="margin-top: 0.5em;">Consensus: <strong>{html.escape(str(agree_ex["consensus"]))}</strong> '
    f'(tier: {html.escape(str(agree_ex["triage_tier"]))})</p>'
    f'</div>'
))

In [ ]:
# Act 3: consensus-tier donut (saves tier_donut.png @ 300 dpi)
render_tier_donut(DATA, PREPRINT_FIG)


In [ ]:
# Act 3: entry-pair disagreement heatmap (saves confusion_heatmap.png @ 300 dpi)
render_confusion_heatmap(DATA, PREPRINT_FIG)


In [ ]:
display(sidebar(
    'Deep dive: Why three models?',
    '<p>Single-model classification had lower precision in our early experiments. '
    'A single model might confidently assign an incident to the wrong entry because '
    'of biases in its training data or the phrasing of the prompt.</p>'
    '<p>Three models with majority vote reduces noise the same way a panel of three '
    'judges reduces individual bias. If two of three models agree, we have higher '
    'confidence in the label. The disagree tier (where all three pick different entries) '
    'explicitly marks incidents where no classifier consensus exists.</p>'
))

display(sidebar(
    'Deep dive: The two-stage classification pipeline',
    '<p><strong>Stage 1 (heuristic):</strong> Regex and keyword indicators scan the '
    'incident text for known patterns (e.g., "prompt injection," "CVE-2026-*"). '
    'This produces a fast initial assignment at low confidence (10%). All incidents '
    'proceed to Stage 2 regardless of Stage 1 results.</p>'
    '<p><strong>Stage 2 (LLM):</strong> Each of three models reads the full incident '
    'text alongside the complete rubric (all 20 entries with inclusion/exclusion '
    'criteria). The model returns an entry_id, a confidence score, and a rationale. '
    'The three Stage 2 labels are combined into the consensus and triage tier.</p>'
))

## Act 4: How Good Is the Classifier?

The classifier is a tool, not ground truth. To trust the incident counts, we need
to measure how often the classifier gets it right — and how often it misses things.

**Precision**: When the classifier says "this incident belongs to LLM02," how often
is it correct? We verified 323 classifications by hand to measure this. Each entry
gets its own precision score — some entries are easier to classify than others.

**Recall**: Does the classifier find all incidents of a given type, or does it miss
some? A human reviewer adjudicated 1,200 incidents across all tiers to measure this.
The reviewer saw the incident text and the three model votes, then decided whether
to accept the consensus, override it, or mark the incident as out of scope.

### Why precision varies so much across entries

The chart below shows precision ranging from 93% (LLM01, LLM03) down to 13% (LLM08).
The variation is not random — it reflects how cleanly each entry's definition separates
it from neighboring categories. Four entries fall **below the 50% threshold**, which
deserves explanation:

- **LLM08 (Vector and Embedding Weaknesses): 13%.** This is the lowest precision in
  the taxonomy. Out of every 8 incidents the classifier labels as LLM08, only 1 actually
  is. The category describes a narrow class of attacks — adversarial manipulation of
  embedding spaces, vector database poisoning, retrieval-augmented generation exploits.
  But the classifier confuses it with LLM03 (Training Data Poisoning) and general data
  integrity issues. Most incidents classified here describe data manipulation that
  affects a model, which is conceptually adjacent but taxonomically distinct.

- **LLM07 (System Prompt Leakage): 31%.** The classifier struggles to distinguish
  "extracting a system prompt" (LLM07) from "overriding a system prompt" (LLM01,
  Prompt Injection). Both involve adversarial interaction with the prompt layer, and
  many real incidents involve both — an attacker extracts the system prompt *in order
  to* craft a better injection. The boundary is taxonomically clear but operationally
  blurred.

- **ROLL-CFAS (Comprehensive Framework Attacks): 33%.** Only 3 precision observations
  — the posterior is dominated by the Beta(1,1) prior, so the 33% estimate has a 90%
  CI stretching from 3% to 78%. The category's broad definition ("attacks on the
  comprehensive AI framework") makes it a natural catch-all that absorbs incidents
  from adjacent entries.

- **ROLL-CMSB (Cross-Modal Safety Bypass): 44%.** This entry sits at the center of the
  confusion boundary described in Act 9B. A deepfake video that bypasses content filters
  could be classified as ROLL-CMSB (cross-modal bypass), LLM09 (misinformation), or
  NEW-WLA (weaponized abuse). The classifier picks one; a human might reasonably pick
  any of the three.

### What the 50% threshold means

Precision below 50% means the classifier is **wrong more often than it is right** for
that entry. When you see an incident labeled "LLM08," the odds are 7:1 against it
actually being a vector/embedding weakness. This has direct consequences for the
Bayesian model in Act 5: low-precision entries get large upward corrections (because
much of their observed count is misclassification noise from other entries) and wide
credible intervals (because the correction itself is uncertain). A 13% precision
estimate does not mean the entry is unimportant — it means the *automated measurement*
of that entry is unreliable, and the model's uncertainty reflects that.

**Stratum coverage note:** The 323 precision verifications were drawn entirely from
the security stratum (CVE/GHSA/OSV). The ai-harm stratum (AIAAIC) has no precision
measurements — the Bayesian model uses a flat Beta(1,1) prior for ai-harm precision,
meaning it assumes no prior knowledge about classifier accuracy on those incidents
(prior mean 0.5). This means error correction for ai-harm incidents relies on
borrowed estimates, not direct measurement.

In [ ]:
# Act 4: classifier precision bars (saves precision_bars.png @ 300 dpi)
render_precision_bars(DATA, PREPRINT_FIG)


In [ ]:
# Act 4: precision Beta posteriors (saves precision_posteriors.png @ 300 dpi)
render_precision_posteriors(DATA, PREPRINT_FIG)


In [ ]:
# precision_data was defined in the Act 4 chart cell (now a library call);
# recompute it here so this sidebar cell is self-contained.
precision_data = DATA['posteriors']['precision']

display(sidebar(
    'Deep dive: The gold-set process — 1,200 human adjudications',
    '<p>A human reviewer (the project author) adjudicated 1,200 incidents using a '
    'blind-first protocol. For each incident:</p>'
    '<ol>'
    '<li>Read the incident text without seeing the model votes (blind label).</li>'
    '<li>Record an independent classification.</li>'
    '<li>Then reveal the three model votes and the consensus.</li>'
    '<li>Make a final decision: accept the consensus, override to a different entry, '
    'assign multiple labels, or mark as out of scope.</li>'
    '</ol>'
    '<p>"Adjudication" is different from voting. The reviewer is not adding a fourth '
    'opinion — they are making a judgment call after seeing both the text and the '
    'model reasoning. The blind label (step 2) guards against anchoring to the '
    'model consensus.</p>'
))

# Full precision posteriors table
prec_table_rows = []
for key, params in sorted(precision_data.items()):
    entry_id = key.split('::')[0]
    if entry_id == 'out-of-scope':
        continue
    alpha, beta_p = params['alpha'], params['beta']
    mean = alpha / (alpha + beta_p)
    ci_low, ci_high = stats.beta.ppf([0.05, 0.95], alpha, beta_p)
    n = int(alpha + beta_p - 2)
    flag = '(prior-dominated)' if n < 5 else ''
    prec_table_rows.append({
        'Entry': entry_id,
        'alpha': alpha, 'beta': beta_p,
        'Mean': f'{mean:.1%}',
        '90% CI': f'[{ci_low:.1%}, {ci_high:.1%}]',
        'n': n,
        'Note': flag,
    })

prec_table_html = pd.DataFrame(prec_table_rows).to_html(index=False, escape=False)
display(sidebar('Deep dive: Full precision posteriors table', prec_table_html))

## Act 5: From Counts to Rankings — The Bayesian Model

Raw incident counts would be misleading. An entry whose classifier has 30% precision
looks like it has many incidents — but two-thirds of those are misclassifications
wrongly attributed to it.

We need a model that adjusts the observed counts for known classifier error. Think
of a bathroom scale that reads 2 pounds heavy. You would subtract 2 pounds from every
reading. The Bayesian model does this for each entry separately, and it carries the
uncertainty through — if the scale is 2±1 pounds off, the corrected weight is also
uncertain.

### What happens with low-precision entries

For entries above 50% precision, the correction is a moderate downward adjustment —
some of the observed incidents were misclassified, so the true count is lower than
the raw count. The correction tightens toward the true signal.

For entries **below 50% precision**, the correction works differently. If only 13%
of incidents labeled "LLM08" actually belong there, the model must infer the true rate
from a signal that is mostly noise. It's like reading a bathroom scale that is off by
more than half the measurement — the "correction" is larger than the reading itself.
This produces two effects: (1) the corrected estimate can be very different from the
raw count, and (2) the uncertainty around that estimate is wide, because small changes
in the precision estimate propagate into large changes in the corrected rate.

This is why some entries in the Act 6 chart have 90% credible intervals spanning 10+
rank positions. The width is not a flaw in the model — it is the model honestly
reporting how much information the data contains about each entry's true incident rate.

### The model's inputs

The model takes the observed incident counts, the measured precision and recall for
each entry, and produces a **posterior distribution** over the true incident rate for
each entry. A posterior distribution is not a single number — it is a range of
plausible values given the data. Wide distributions mean less certainty.

We drew 16,000 samples from this distribution using Markov chain Monte Carlo (MCMC).
MCMC is a method for sampling from probability distributions that are too complex to
compute directly. It generates a sequence of random samples that, after enough
iterations, represents the target distribution. Our run used 4 chains of 4,000
samples each, with 2,000 warmup iterations per chain.

### Handling missing data

Three entries — LLM04, LLM08, LLM10 — are **frame-blind**: their incident counts
come entirely from one stratum, so the model cannot cross-validate their rates across
strata. These entries are included in the posterior but flagged with a ★ in the charts.
Their rank estimates carry additional structural uncertainty beyond what the credible
intervals capture.

**For 16 of 20 entries in the ai-harm stratum, we have not measured recall
directly.** The model uses a conservative prior estimate of ~1% recall for those
entries — Beta(1, 101). This means the model assumes the classifier finds very few
of those incidents and adjusts upward accordingly. These corrections are large, which
is one reason the credible intervals in Act 6 are wide.

**Precision in the ai-harm stratum is also unmeasured.** The model uses a flat
Beta(1,1) = Uniform(0,1) prior, meaning it treats ai-harm precision as completely
unknown (prior mean 50%). This is a weaker assumption than the security stratum,
where we have 5–88 hand-verified observations per entry.

In [ ]:
# Act 5: posterior lambda ridge plot (saves ridge_plot.png @ 300 dpi)
render_ridge_plot(DATA, PREPRINT_FIG)


In [ ]:
# lambda_samples was defined in the Act 5 chart cell (now a library call).
lambda_samples = DATA['lambda_samples']

# Summary statistics from lambda_samples and diagnostic.json
summary_rows = []
for i, eid in enumerate(INFER_ENTRY_ORDER):
    samples = lambda_samples[:, i]
    med = np.median(samples)
    ci_low, ci_high = np.percentile(samples, [5, 95])
    diag_report = DATA['diagnostic']['entry_reports'].get(eid, {})
    flag = diag_report.get('flag', '—')

    summary_rows.append({
        'Entry': eid,
        'Name': ENTRY_NAMES.get(eid, ''),
        '_median_numeric': med,
        'Median λ': f'{med:.4f}',
        '90% CI': f'[{ci_low:.4f}, {ci_high:.4f}]',
        'CI Width': f'{ci_high - ci_low:.4f}',
        'Diagnostic': flag,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('_median_numeric', ascending=False)
summary_df = summary_df.drop(columns=['_median_numeric'])
display(summary_df.style
    .set_caption('Posterior summary: median incident rate, 90% credible interval, diagnostic flag')
    .set_properties(**{'text-align': 'left'})
    .apply(lambda row: ['background: #f0f0f0' if row['Entry'] in FRAME_BLIND else ''
                        for _ in row], axis=1)
)

In [ ]:
# NumPyro model specification sidebar
model_source = (REPO_ROOT / 'engine' / 'model' / 'inference.py').read_text()
# Extract just the model function
model_start = model_source.find('def model(')
model_end = model_source.find('\n    # ------', model_start + 1)
if model_end == -1:
    model_end = model_source.find('\n    try:', model_start)
model_code = model_source[model_start:model_end]

display(sidebar(
    'Deep dive: The NumPyro model specification',
    '<p>This is the actual model we used, written in NumPyro (a probabilistic '
    'programming library for JAX). You do not need to install NumPyro to run this '
    'notebook — the results above are pre-computed.</p>'
    f'<pre style="background: #1a202c; color: #e2e8f0; padding: 1em; '
    f'border-radius: 4px; overflow-x: auto; font-size: 0.85em;">'
    f'{model_code.replace("<", "&lt;").replace(">", "&gt;")}</pre>'
    '<p><strong>Key components:</strong></p>'
    '<ul>'
    '<li><code>lambda</code>: latent prevalence per entry (HalfNormal prior)</li>'
    '<li><code>recall</code>, <code>precision</code>: per-entry, per-stratum '
    'measurement error (Beta priors from gold-set calibration)</li>'
    '<li><code>concentration</code>: over-dispersion parameter (Gamma prior)</li>'
    '<li>The FP leakage term (<code>einsum</code>) accounts for misclassified '
    'incidents that spill from one entry into another</li>'
    '<li>Negative-Binomial likelihood handles count over-dispersion</li>'
    '</ul>'
))

# MCMC diagnostics sidebar
inf_summary = DATA['inference_summary']
max_rhat = max(v for k, v in inf_summary['r_hat'].items() if k.startswith('lambda'))
min_ess = min(v for k, v in inf_summary['ess'].items() if k.startswith('lambda'))
total_draws = inf_summary['num_samples'] * inf_summary['num_chains']

display(sidebar(
    'Deep dive: MCMC convergence diagnostics',
    f'<p>The MCMC sampler ran <strong>{inf_summary["num_chains"]} chains</strong>, '
    f'each with <strong>{inf_summary["num_warmup"]:,}</strong> warmup iterations '
    f'and <strong>{inf_summary["num_samples"]:,}</strong> sampling iterations, '
    f'producing <strong>{total_draws:,}</strong> posterior draws total.</p>'
    '<p><strong>R-hat</strong> measures whether the chains converged to the same '
    'distribution. Values near 1.0 mean convergence. Our maximum R-hat across all '
    f'lambda parameters: <strong>{max_rhat:.6f}</strong> (threshold: ≤1.01). '
    'All parameters pass.</p>'
    f'<p><strong>Effective sample size (ESS)</strong> measures how many independent '
    f'samples the chains produced. Our minimum ESS for lambda parameters: '
    f'<strong>{min_ess:,.0f}</strong> out of {total_draws:,} draws '
    f'({min_ess/total_draws:.0%} efficiency). Higher is better.</p>'
    f'<p><strong>Divergences</strong>: <strong>{inf_summary["divergences"]}</strong>. '
    'Zero divergences means the sampler explored the posterior geometry without '
    'numerical problems. Any non-zero count would indicate regions the sampler '
    'could not traverse reliably.</p>'
))

## Act 6: The Incident-Derived Rankings

These rankings reflect what the incident data suggests after correcting for classifier
error. They are one signal, not the final word.

For each entry, the Bayesian model gives us a posterior distribution over its true
incident rate. We rank entries by their median rate and report a 90% credible interval
on the rank. Some entries have tight intervals (the data is informative) and others are
wide (less certain). The width tells you how much to trust the rank position.

### How to read the chart

Each row is one taxonomy entry. The diamond marks the **median rank** — the rank
position at the center of the posterior distribution. The horizontal bar spans the
**90% credible interval** — the range of ranks that the model considers plausible
given the data and its uncertainty about precision, recall, and the true incident rate.

**Tight intervals** (e.g., LLM02 spanning 1–6) mean the data strongly constrains
that entry's position. Even after accounting for classifier error, the evidence
points to a narrow range.

**Wide intervals** (e.g., spanning 6–20) mean the data is compatible with many rank
positions. This happens when the entry has low precision (the correction is large and
uncertain), few observations (small sample sizes produce wide posteriors), or unmeasured
recall (the model uses a conservative prior that adds uncertainty).

**Grey entries** (LLM04, LLM08, LLM10) are frame-blind — their measurements come from
only one stratum. Their positions carry structural uncertainty beyond what the CI
captures.

The static matplotlib chart shows the rank distributions. The interactive plotly
chart below it lets you hover over each entry for details: median rank, CI bounds,
diagnostic flag, and frame-blind status.

In [ ]:
# Act 6: incident-derived rank dumbbell (saves dumbbell_chart.png @ 300 dpi)
render_dumbbell_chart(DATA, PREPRINT_FIG)


In [ ]:
# Act 6: incident-derived rankings, plotly static PNG (saves plotly_rankings.png @ 300 dpi)
render_plotly_rankings(DATA, PREPRINT_FIG)


In [ ]:
# Fill in the incident-rank column from Act 6.  Median incident-derived rank
# per entry = median of per-draw ranks (matches the dumbbell chart).
_lam = DATA['lambda_samples']
_rank_mat = np.argsort(np.argsort(-_lam, axis=1), axis=1) + 1  # 1 = highest rate
_median_rank = {
    eid: float(np.median(_rank_mat[:, i]))
    for i, eid in enumerate(INFER_ENTRY_ORDER)
}

updated_table = pd.DataFrame([
    {
        '#': i + 1,
        'Entry ID': e['entry_id'],
        'Name': e['canonical_name'],
        'Incident Rank': (f"{_median_rank[e['entry_id']]:.0f}"
                          if e['entry_id'] in _median_rank else '\u2014'),
    }
    for i, e in enumerate(DATA['rubric']['entries'])
])
updated_table = updated_table.set_index('#')
display(updated_table.style.set_caption(
    "The table from Act 1, now with incident-derived ranks filled in."
).set_properties(**{'text-align': 'left'}))


## Act 7: The Confrontation — Do Experts and Incidents Agree?

Cohen's weighted kappa measures agreement between two ranking systems, adjusted for
chance. A value of 1.0 means perfect agreement. A value of 0 means no better than
random. Negative values mean systematic disagreement.

**Our result: kappa = 0.20, with a 90% credible interval of [−0.16, 0.57].** This
interval includes zero. We cannot exclude the possibility that expert and incident
rankings agree by chance alone. The point estimate of 0.20 suggests slight agreement,
but the wide interval means this is a weak signal, not a firm conclusion.

Why is the interval so wide? Two reasons. First, we only have 17 measurable entries
(three are frame-blind). Statistical agreement measures need larger samples for narrow
confidence intervals. Second, the posterior rank distributions themselves are wide —
most entries have 90% CIs spanning 10+ rank positions.

Five entries have posterior probability of tier mismatch exceeding 83% — meaning that
across the full joint posterior, the Bayesian model and expert survey place these
entries in different thirds of the ranking more than 83% of the time:

- **LLM01 Prompt Injection**: experts rank it #1 (90% CI: 1–2), incidents rank it #12 (90% CI: 4–18)
- **LLM09 Misinformation**: incidents rank it #2 (90% CI: 1–5), experts rank it #13 (90% CI: 9–16)
- **NEW-MTIE MCP Tool Interface Exploitation**: experts rank it #7 (90% CI: 5–9), incidents #16 (90% CI: 6–20)
- **NEW-PMP Persistent Memory Poisoning**: experts rank it #4 (90% CI: 2–7), incidents #16 (90% CI: 6–20)
- **NEW-WLA Weaponized LLM Abuse**: incidents rank it #8 (90% CI: 3–15), experts #17 (90% CI: 13–20)

In [ ]:
# Act 7: expert-vs-incident bump/slope chart (saves bump_chart.png @ 300 dpi)
render_bump_chart(DATA, PREPRINT_FIG)


In [ ]:
# Act 7: rank CI overlap (saves ci_overlap.png @ 300 dpi)
render_ci_overlap(DATA, PREPRINT_FIG)


In [ ]:
display(sidebar(
    'Deep dive: How weighted kappa works',
    '<p>Cohen\'s kappa compares observed agreement to expected agreement by chance. '
    'Weighted kappa extends this to ordinal data — disagreements by 1 tier are '
    'penalized less than disagreements by 2+ tiers.</p>'
    '<p>Concretely: we divide the 20 entries into rank tiers (top 5, 6-10, 11-15, '
    '16-20). Each entry gets a tier from the expert ranking and a tier from the '
    'incident ranking. Kappa measures how often these tiers match, minus what we '
    'would expect from random assignment.</p>'
    '<p>With only <strong>17 measurable entries</strong> (three are frame-blind and excluded), '
    'the sample size is small. Small samples produce wide confidence intervals. '
    'A kappa of 0.20 on 17 observations could easily be consistent with true kappa '
    'values anywhere from -0.16 to 0.57.</p>'
    '<p>For reference: kappa &lt; 0 = worse than chance, 0-0.20 = slight, '
    '0.21-0.40 = fair, 0.41-0.60 = moderate, &gt; 0.60 = substantial.</p>'
))

sb = DATA['selection_bias']
display(sidebar(
    'Deep dive: Selection bias test',
    f'<p>We tested whether incident rates differ systematically between strata '
    f'using a Kruskal-Wallis test (a non-parametric test for differences across groups).</p>'
    f'<p>Result: H = {sb["statistic_value"]:.3f}, p = {sb["p_value"]:.3f}, '
    f'severity = {sb["severity"]}.</p>'
    f'<p>A p-value of {sb["p_value"]:.2f} means we cannot reject the null hypothesis '
    f'that incident rates are similar across strata. In plain language: the security '
    f'and ai-harm strata do not show statistically different patterns of entry prevalence. '
    f'This is reassuring — it means our results are not driven by one stratum dominating.</p>'
))

## Act 8: Where Experts and Incidents Disagree

Five entries have notable tier mismatches. For each, we dig into *why* the disagreement
exists — what the data shows and what it might mean.

**LLM01 (Prompt Injection)**: Expert #1, incident #12. Prompt injection is the
best-understood LLM attack. Deployed systems defend against it actively — input
filtering, output sandboxing, system prompt hardening. Fewer incidents reach public
databases because defenses often work. Experts rank it #1 because the attack surface
is enormous even when defenses hold. The incident data sees fewer successful exploits,
so the model ranks it lower.

**LLM09 (Misinformation)**: Incident #2, expert #13. The corpus contains a large
volume of deepfake and AI-generated disinformation incidents from the AIAAIC harm
database. Experts may rank misinformation lower because "misinformation" as a category
overlaps with other entries (NEW-WLA, ROLL-CMSB) and because many of these incidents
describe harm *from* AI rather than a vulnerability *in* an LLM. We will examine this
overlap in Act 9B.

**NEW-PMP (Persistent Memory Poisoning)** and **NEW-MTIE (MCP Tool Interface
Exploitation)**: Expert top-5, almost no incidents yet. These are emerging threats —
persistent memory poisoning and MCP tool exploitation are new enough that the public
incident record has not caught up. If the goal is to warn practitioners, expert signal
may matter more than incident counts for emerging threats.

**NEW-WLA (Weaponized LLM Abuse)**: 863 incidents, expert rank 17. The large incident
count is driven by a broad entry definition that captures AI-generated disinformation,
deepfake CSAM, and synthetic media abuse. Experts may rank it low because many of
these incidents describe harm *from* AI systems rather than an exploitable
vulnerability *in* an LLM.

In [ ]:
# Act 8: flagged-entry paired dots (saves paired_dots.png @ 300 dpi)
render_paired_dots(DATA, PREPRINT_FIG)


In [ ]:
# Act 8: incident-theme keyword bars for LLM09 and NEW-WLA (300-dpi PNGs)
render_theme_bars(DATA, PREPRINT_FIG, 'LLM09', 'theme_bars_llm09.png')
render_theme_bars(DATA, PREPRINT_FIG, 'NEW-WLA', 'theme_bars_new_wla.png')


## Act 9: What the Data Cannot See

The ranking analysis covers the 17 measurable entries. But two patterns in the data
reveal structural limits of what incident-counting can tell us.

### 9A: "AI Harm Without LLM Vulnerability"

2,394 incidents — about 40% of the corpus — landed in "out of scope." All three
models agreed these do not belong to any of the 20 taxonomy entries.

These are real AI harms. Facial recognition that misidentifies people. Algorithmic
hiring tools that discriminate. Drones used for surveillance. Recommendation engines
that radicalize users. But none of them describe a vulnerability *in* a large language
model. They are incidents *from* AI systems, not incidents *of* LLM vulnerabilities.

This gap is a feature of the sampling frame, not a failure of the taxonomy. The corpus
was built by crawling CVE/GHSA/OSV databases with AI-related keywords. Those keywords
pull in any incident that mentions "AI" or "machine learning," regardless of whether
an LLM is involved. The AIAAIC harm database, by design, covers all AI-related harms.

The out-of-scope cluster matters because it shows the boundary of what this methodology
can measure. Incident-counting works when incidents map to taxonomy entries. For harms
that sit outside the taxonomy — because they involve non-LLM AI, or because they
describe societal effects rather than technical vulnerabilities — the incident signal
is silent.

In [ ]:
# Act 9A: out-of-scope theme treemap, plotly static PNG (saves oos_treemap.png @ 300 dpi)
render_oos_treemap(DATA, PREPRINT_FIG)


In [ ]:
# Act 9A: which disagree/split-tier vote combinations the human reviewer sent
# to out-of-scope.  Text summary only (not one of the preprint figures).
prelabel_lookup = {p['incident_id']: p for p in DATA['prelabels']}
disagree_oos_votes = []
for g in DATA['goldset']:
    if (g['llm_consensus'] == 'out-of-scope'
            and g['adjudicated'] == 'accept'
            and g['incident_id'] in prelabel_lookup):
        pl = prelabel_lookup[g['incident_id']]
        if pl['triage_tier'] in ('disagree', 'split'):
            votes = tuple(sorted(v['entry_id'] for v in pl['model_votes']))
            disagree_oos_votes.append(votes)

top_combos = Counter(disagree_oos_votes).most_common(10)
if top_combos:
    print("Disagree/split-tier vote combinations the reviewer sent to out-of-scope:")
    for combo, n in top_combos:
        print(f"  {n:3d}  {' / '.join(combo)}")
else:
    print("No disagree/split-tier incidents were adjudicated to out-of-scope.")


### 9B: The LLM09 / NEW-WLA / ROLL-CMSB Confusion Boundary

A **confusion boundary** is a region where categories overlap enough that classifiers
— and sometimes humans — cannot reliably tell them apart. The problem is not that the
classifier is broken. The problem is that the categories share real conceptual
territory.

The data shows a clear confusion boundary between three entries:

- **LLM09 (Misinformation)**: the output is false or misleading
- **NEW-WLA (Weaponized LLM Abuse)**: an adversary uses AI as a weapon
- **ROLL-CMSB (Cross-Modal Safety Bypass)**: the attack uses image/video/audio modalities

Consider a deepfake video that spreads political disinformation. Which entry does it
belong to? It is misleading content (LLM09). It was created using AI as a weapon
(NEW-WLA). It exploits an image/video generation modality (ROLL-CMSB). The three
categories overlap in real-world incidents, and the overlap is not a classification
error — it reflects genuine ambiguity in the taxonomy.

This matters for interpretation. When the incident data ranks LLM09 at #2, some of
that signal comes from incidents that could equally have been classified as NEW-WLA or
ROLL-CMSB. The confusion boundary inflates counts for whichever entry the classifier
happens to prefer and deflates counts for the others. The Bayesian model corrects for
measured precision (how often each entry's classifications are right), but it cannot
correct for ambiguity that the gold-set reviewers themselves found difficult to resolve.

In [ ]:
# Act 9B: confusion-boundary Sankey, plotly static PNG (saves sankey_confusion.png @ 300 dpi)
render_sankey_confusion(DATA, PREPRINT_FIG)


In [ ]:
# Act 9B: 3x3 confusion matrix (saves confusion_matrix_3x3.png @ 300 dpi)
render_confusion_matrix_3x3(DATA, PREPRINT_FIG)


In [ ]:
# boundary_list was defined in the Act 9B chart cell (now a library call).
boundary_list = ['LLM09', 'NEW-WLA', 'ROLL-CMSB']

# Show 2-3 real incidents from the confusion boundary
boundary_examples = []
for p in DATA['prelabels']:
    if p['triage_tier'] == 'disagree':
        votes = set(v['entry_id'] for v in p['model_votes'])
        if votes & {'LLM09', 'NEW-WLA', 'ROLL-CMSB'} and len(votes & set(boundary_list)) >= 2:
            boundary_examples.append(p)
    if len(boundary_examples) >= 3:
        break

# Fall back to split tier if not enough disagree examples
if len(boundary_examples) < 2:
    for p in DATA['prelabels']:
        if p['triage_tier'] == 'split':
            votes = set(v['entry_id'] for v in p['model_votes'])
            if len(votes & set(boundary_list)) >= 2:
                boundary_examples.append(p)
        if len(boundary_examples) >= 3:
            break

display(HTML('<h4>Real incidents from the confusion boundary</h4>'))
for ex in boundary_examples[:3]:
    text = html.escape(shorten(ex['text'], width=300, placeholder='...'))
    vote_html = ''.join(
        f'<li><strong>{html.escape(v["model_id"].split("/")[-1])}</strong>: '
        f'{html.escape(v["entry_id"])} ({v["confidence"]:.0%})</li>'
        for v in ex['model_votes']
    )

    # Check if this incident was adjudicated
    adj = next((g for g in DATA['goldset'] if g['incident_id'] == ex['incident_id']), None)
    adj_html = ''
    if adj:
        labels_str = html.escape(", ".join(adj["labels"]) if adj["labels"] else "out-of-scope")
        notes_str = html.escape(adj["notes"]) if adj.get("notes") else ""
        adj_html = (f'<p><strong>Human decision:</strong> {html.escape(adj["adjudicated"])} → '
                    f'{labels_str}'
                    f'{" — " + notes_str if notes_str else ""}</p>')

    display(HTML(
        f'<div style="border: 1px solid #805ad5; border-radius: 8px; padding: 1em; '
        f'margin: 0.5em 0; background: #faf5ff;">'
        f'<code>{html.escape(ex["incident_id"])}</code> · tier: '
        f'{html.escape(ex["triage_tier"])}<br>'
        f'<p style="color: #333; font-style: italic;">{text}</p>'
        f'<ul style="margin: 0.5em 0;">{vote_html}</ul>'
        f'{adj_html}'
        f'</div>'
    ))

## Act 10: What This Means

**Where the data and experts agree.** LLM02 (Sensitive Information Disclosure) sits
near the top by both measures — experts rank it #2 and incidents rank it #2. ROLL-SICG,
NEW-ITSCD, and NEW-MSDA are consistently near the bottom by both signals. These
positions are stable across the uncertainty ranges.

**Where the data pushes back.** LLM09's incident volume is much higher than its expert
rank. Part of this comes from the broad entry definition, which captures AI-adjacent
harms (deepfake misuse, synthetic disinformation) that may not represent LLM
vulnerabilities in the narrow sense. NEW-WLA shows a similar pattern. The confusion
boundary between LLM09, NEW-WLA, and ROLL-CMSB inflates whichever entry the classifier
prefers and makes all three counts less reliable than entries with cleaner boundaries.

**What the experts see that incidents miss.** NEW-PMP and NEW-MTIE have almost no
incidents in the public record but strong expert signal. These are forward-looking
entries — persistent memory poisoning and MCP tool exploitation are new enough that the
incident databases have not caught up. If the purpose of the Top 10 is to warn
practitioners about threats they will face, expert signal matters more than incident
counts for emerging threats.

**What this methodology can and cannot do.** This is a triangulation tool. It checks
one signal (expert surveys) against another (incident data). Neither signal is the
truth. The incident data has known structural biases: the sampling frame misses
incidents that are not publicly reported, the classifier has measured error rates, and
the taxonomy-frame circularity (F-circ) means we are partially measuring the
classifier's preferences rather than the true threat distribution. The expert data has
its own biases: availability bias, recency effects, anchoring to prior Top 10 lists.

The value is in the comparison, not in either signal alone. Where experts and incidents
agree, confidence is higher. Where they diverge, the disagreement itself is the finding
— it points to entries where one signal or the other may be systematically distorted.

**The kappa ceiling is structural.** Some of the disagreement between expert and
incident rankings is informative — it reveals real differences between "what experts
worry about" and "what has actually happened." Perfect agreement would be surprising
and arguably suspicious. The current kappa of 0.20 [−0.16, 0.57] is consistent with
weak-to-moderate agreement, but the confidence interval is too wide to draw firm
conclusions. A larger corpus, better-defined entry boundaries, and independent recall
measurement would all narrow the interval and sharpen the comparison.

**Accepted limitation: ai-harm precision.** The 323 precision verifications were drawn entirely from the security stratum. The ai-harm stratum (92 in-scope incidents across 8 entry assignments, of which only 3 received recall posteriors with material evidence — LLM09, LLM04, NEW-MA; NEW-WLA has only 1 observation above the pure prior) has no direct precision measurements — ai-harm precision keys are absent from the calibration data entirely. The model falls back to a flat Beta(1,1) = Uniform(0,1) prior for ai-harm precision, meaning it assumes no prior knowledge about how precise the classifier is on ai-harm incidents (prior mean 0.5). Closing this gap would require sourcing additional ai-harm incidents beyond the existing corpus, which is outside this project's scope. The disclosure in Acts 4 and 5 describes how the model handles missing precision data.

## Act 11: Robustness Under Frontier Classifiers

<!-- PLACEHOLDER — prose author fills this section; no narrative in the code pass. -->


In [ ]:
# Act 11: RARR robustness — ranking fidelity (Spearman rho vs held-out truth)
# for each frontier classifier and the ensemble, against the incidence floor.
# Saves rarr_robustness.png @ 300 dpi.
_rarr_robustness = json.loads(
    (RARR_CYCLE / 'results' / 'robustness_validation.json').read_text()
)
render_rarr_robustness(_rarr_robustness, PREPRINT_FIG)


## The 2026 Blended Top 10

<!-- PLACEHOLDER — prose author fills this section; no narrative in the code pass. -->


In [ ]:
# The 2026 candidate ballot: 10 incumbents (LLM01-LLM10) + 6 NEW-* + 4 ROLL-*,
# with arrows from each roll-up to the incumbent it folds into.
# Saves entry_expansion_map.png @ 300 dpi.
_entries = load_entries(CYCLE / 'taxonomy' / 'taxonomy.json')
render_entry_expansion_map(_entries, PREPRINT_FIG)


In [ ]:
# Blended 2026 Top 10 (Scope-1 fold, 0.75*vote + 0.25*lambda) and its movement
# from the published 2025 order.  Saves rank_change_2025_2026.png @ 300 dpi.
#
# Derivation (see docs/BLENDED-TOP10-METHODOLOGY.md and the task report):
#   * vote_rank / lambda_rank per entry = median columns of the committed
#     results/rank_comparison_report.md (the methodology doc's stated source).
#     The raw respondent_rankings.npy median does NOT reproduce the doc's
#     aggregated 20-field vote ranks (e.g. LLM07 = 11.5), so we read the frozen
#     report the doc itself cites.
#   * fold rule = max-severity (min rank) of each incumbent and the child that
#     rolls into it, on each axis, using taxonomy.json's rolled_into crosswalk.
#   * blend + ordering + moves via the tested engine.report.blend_2025_2026.
# This reproduces the methodology doc's Section 5 movers:
#   Improper Output Handling -5, Unbounded Consumption +4, Excessive Agency +3.
_rank_md = DATA['rank_comparison_md']
_lam_rank, _vote_rank = {}, {}
for _line in _rank_md.splitlines():
    if _line.startswith('|') and 'Entry' not in _line and not _line.startswith('|--'):
        _cols = [c.strip() for c in _line.split('|')[1:-1]]
        if len(_cols) >= 3:
            _m1 = re.match(r'([\d.]+)', _cols[1])
            _m2 = re.match(r'([\d.]+)', _cols[2])
            if _m1 and _m2:
                _lam_rank[_cols[0]] = float(_m1.group(1))
                _vote_rank[_cols[0]] = float(_m2.group(1))

_entries = load_entries(CYCLE / 'taxonomy' / 'taxonomy.json')
_incumbents = [e['entry_id'] for e in _entries if e['group'] == 'incumbent']
_child_of = {
    e['entry_id']: e['rolled_into'] for e in _entries if e['group'] == 'rollup'
}

_folded_lam = dict(_lam_rank)
_folded_vote = dict(_vote_rank)
for _child, _parent in _child_of.items():
    if _parent in _folded_lam and _child in _lam_rank:
        _folded_lam[_parent] = min(_folded_lam[_parent], _lam_rank[_child])
        _folded_vote[_parent] = min(_folded_vote[_parent], _vote_rank[_child])

_vote_ranks = {e: _folded_vote[e] for e in _incumbents}
_lambda_ranks = {e: _folded_lam[e] for e in _incumbents}
blended = blended_ranking(_vote_ranks, _lambda_ranks, w_vote=0.75)
_published_order = [f'LLM{i:02d}' for i in range(1, 11)]
_moves = rank_moves(_published_order, blended)

render_rank_change_2025_2026(blended, ENTRY_NAMES, PREPRINT_FIG)

_biggest = sorted(_moves.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
print("Blended 2026 Top 10 (rank / entry / blend / move-from-published):")
for _b in blended:
    print(f"  {_b['blend_rank']:2d}  {_b['entry_id']:6s} "
          f"blend={_b['blend']:.3f}  move={_moves[_b['entry_id']]:+d}")
print("Biggest movers: " + ", ".join(
    f"{ENTRY_NAMES.get(e, e)} {m:+d}" for e, m in _biggest))


In [ ]:
# Consistency check: guard the two load-bearing numbers cited across the preprint.
_rob = json.loads((RARR_CYCLE / 'results' / 'robustness_validation.json').read_text())
_floor_rho = _rob['ranking_fidelity_spearman_vs_truth']['floor']
assert _floor_rho > 0.85, f"floor Spearman rho {_floor_rho} not > 0.85"

_baselines = json.loads((BASELINES / 'rankings_baselines.json').read_text())
_kappa = _baselines['previous_ranking']['kappa_median']
assert _kappa == 0.2028985507246377, f"kappa {_kappa} != 0.2028985507246377"

print("consistency OK")


## Glossary

<!-- PLACEHOLDER — prose author fills this section; no narrative in the code pass. -->


## Limitations

<!-- PLACEHOLDER — prose author fills this section; no narrative in the code pass. -->


## Scope

<!-- PLACEHOLDER — prose author fills this section; no narrative in the code pass. -->
